In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import torch

import sys
sys.path.insert(0, '../../../')

from src.difsched.config import getExpConfig, visualizeExpConfig
from src.difsched.env.Hybrid import createEnv
from src.difsched.utils.EnvInterface import EnvInterface
from src.difsched.evaluation import loadAndEvaluation

In [2]:
datasetFolder = f'../../../data/processed/offline_dataset'

In [3]:
results = []

for expConfigIdx in range(7,8):
    expParams = getExpConfig(expConfigIdx)
    visualizeExpConfig(expParams)

    dataset_off = {
        'observations': [],
        'actions': [],
        'rewards': [],
        'next_observations': []
    }
    for exp_idx in expParams['offline_dataset_idxs']:
        with open(f'{datasetFolder}/subOptimalAgent_envConfig{exp_idx}.pkl', 'rb') as f:
            dataset_expert = pickle.load(f)
        
        dataset_off['observations'].extend(dataset_expert['uRecord'])
        dataset_off['actions'].extend(dataset_expert['actionsRecord'])
        dataset_off['rewards'].extend(dataset_expert['rewardRecord'])
        dataset_off['next_observations'].extend(dataset_expert['uNextRecord'])
        
        print(f"Exp {exp_idx} - Avg. packet loss rate: {np.mean(dataset_expert['rewardRecord'])}")
        print(f"Exp {exp_idx} - length of dataset: {len(dataset_expert['uRecord'])}")

    print(f"\nCombined dataset length: {len(dataset_off['observations'])}")
    print(f"Combined avg. packet loss rate: {np.mean(dataset_off['rewards'])}")

    trafficDataParentPath = f'../../../data/processed/traffic'
    env = createEnv(expParams, trafficDataParentPath)
    env.selectMode(mode="test", type="data")
    envInterface = EnvInterface(expParams, discrete_state=False)

    modelFolder = f'../../../data/results/dql/config_{expConfigIdx}'
    eval_results = loadAndEvaluation(env, envInterface, dataset_expert, modelFolder, exp_idx_list=[0], eta=0.001)
    
    for eval_res in eval_results:
        results.append({
            'Exp_ID': expConfigIdx,
            'N_user': expParams['N_user'],
            'dataflow': expParams['dataflow'],
            'LEN_window': expParams['LEN_window'],
            'N_aggregation': expParams['N_aggregation'],
            'r_bar': expParams['r_bar'],
            'B': expParams['B'],
            'Model_idx': eval_res['model_idx'],
            'DQL_Reward': eval_res['reward_mean'],
            'DQL_Reward_Std': eval_res['reward_std'],
            'DQL_PacketLoss': eval_res['packet_loss']
        })

df_results = pd.DataFrame(results)


EnvType: HYBRID
N_user: 32
LEN_window: 100
N_aggregation: 4
dataflow: haptic_1ms_100
randomSeed: 999
r_bar: 4
B: 180
sigma_list: [0.7, 0.75, 0.8, 0.85, 0.9]
offline_dataset_idxs: [21, 22, 23]
Exp 21 - Avg. packet loss rate: 0.014529115204818553
Exp 21 - length of dataset: 10000
Exp 22 - Avg. packet loss rate: 0.013207397704001133
Exp 22 - length of dataset: 10000
Exp 23 - Avg. packet loss rate: 0.013325415527793129
Exp 23 - length of dataset: 10000

Combined dataset length: 30000
Combined avg. packet loss rate: 0.01368730947887094
Expert's Reward: 0.013325415527793129
Loading model 0_best


reward_diffusionQ0: mean=0.0217, std=0.0121


In [4]:
print("\n" + "="*100)
print("EVALUATION RESULTS SUMMARY")
print("="*100 + "\n")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

summary_df = pd.DataFrame({
    'ID': df_results['Exp_ID'],
    'dataflow': df_results['dataflow'],
    'N_user': df_results['N_user'],
    'DQL_PacketLoss': df_results.apply(lambda row: f"{row['DQL_PacketLoss']:.4f}±{row['DQL_Reward_Std']:.4f}", axis=1)
})

summary_df = summary_df.sort_values(by=['dataflow', 'N_user'])

print(summary_df.to_string(index=False))



EVALUATION RESULTS SUMMARY

 ID       dataflow  N_user DQL_PacketLoss
  7 haptic_1ms_100      32  0.0217±0.0121
